In [ ]:
# CELL 0 - CONFIG
# Fill these in for your own environment. Nothing below this cell should need
# touching for a normal run. Defaults below match a Kaggle notebook with a
# private dataset attached.

import os

# --- Where things live ---
WORKDIR = "/kaggle/working"                                           # writable scratch space
DATASET_DIR = "/kaggle/input/datasets/<your-username>/<your-dataset>" # your model dataset mount
LLAMA_CPP_SRC = f"{DATASET_DIR}/llama.cpp-b9775-cuda-12.8-amd64/cuda-12.8"  # your llama.cpp CUDA build
MODEL_DIR = DATASET_DIR                                               # where GGUF/safetensors live

# --- llama.cpp shared-library versions ---
# Only matters if Cell 2's symlink step throws or the server refuses to start.
# Check what your build actually ships with `ls` inside LLAMA_CPP_SRC and
# update these two strings to match.
LLAMA_LIB_VERSION = "0.0.1"
GGML_LIB_VERSION = "0.15.2"

# --- Model filenames (relative to MODEL_DIR) ---
MODEL_TALK = "L3.2-Rogue-Creative-Instruct-Uncensored-7B-D_AU-q5_k_m.gguf"
MODEL_VISION = "Qwen3.5-9B-Claude-4.6-OS-AV-H-UNCENSORED-THINK-D_AU-Q5_K_M-imat.gguf"
MMPROJ_VISION = "mmproj-F16(9B).gguf"
MODEL_DEEP = "Qwen3.6-40B-Deck-Opus-NEO-CODE-HERE-2T-OT-Q4_K_M.gguf"
MMPROJ_DEEP = "mmproj-F16.gguf"
MODEL_CYBERREALISTIC = "CyberRealistic_V4.1_FP16.safetensors"
MODEL_REALVISXL = "RealVisXL_V5.0_Lightning_fp16.safetensors"

# --- Binary + library locations ---
# Defaults assume Cells 1-3 ran and built a working copy at WORKDIR/llama.
# Running locally (VS Code, plain Jupyter, your own box) with a normal
# llama.cpp install? Skip Cells 1-3 entirely and point these two straight at
# your own binary instead -- most local CUDA installs need LIB_PATHS = [].
SERVER_BIN = f"{WORKDIR}/llama/llama-server"
LIB_PATHS = [f"{WORKDIR}/llama", "/usr/local/nvidia/lib64", "/usr/local/cuda/lib64"]

# --- Ports ---
PORT_TALK = 8081
PORT_VISION = 8082
PORT_DEEP = 8083
PORT_WEBUI = 3000
PORT_IMAGE = 7860

# --- Open WebUI ---
# False means anyone with your tunnel link gets in with no login. Fine for a
# five-minute personal session, worth turning on if this repo is public and
# someone else is running it as a standing setup.
WEBUI_REQUIRE_LOGIN = True

# --- Platform ---
# Kaggle mounts your dataset read-only, so the build has to be copied into a
# writable dir first. On your own machine / another cloud box your files are
# probably already writable, so set this False and skip the copy entirely.
RUNNING_ON_KAGGLE = True

# The third-party llama.cpp CUDA build I used shipped with broken shared-
# library symlinks. If `ls` inside LLAMA_CPP_SRC shows versioned .so files
# with no matching unversioned name, leave this True. If your build already
# has normal symlinks (most official releases do), set it False.
FIX_SYMLINKS = True

# Where CUDA/the NVIDIA driver libs live on your system, in addition to the
# llama.cpp build dir itself. These two are Kaggle's paths. On a local Ubuntu
# box this is often just "/usr/local/cuda/lib64" or "/usr/lib/x86_64-linux-gnu".
CUDA_LIB_PATHS = ["/usr/local/nvidia/lib64", "/usr/local/cuda/lib64"]

os.makedirs(WORKDIR, exist_ok=True)
print("CONFIG LOADED")
print(f"WORKDIR: {WORKDIR}")
print(f"MODEL_DIR: {MODEL_DIR}")


In [ ]:
# CELL 1
import shutil
import os

if RUNNING_ON_KAGGLE:
    LLAMA_DIR = f"{WORKDIR}/llama"
    if os.path.exists(LLAMA_DIR):
        shutil.rmtree(LLAMA_DIR)
    shutil.copytree(LLAMA_CPP_SRC, LLAMA_DIR)
    print("COPIED")
else:
    LLAMA_DIR = LLAMA_CPP_SRC
    print(f"Using llama.cpp build in place at {LLAMA_DIR}")


In [ ]:
# CELL 2
import os
os.chdir(LLAMA_DIR)

if FIX_SYMLINKS:
    LIBS = [
        ("libllama-common.so", LLAMA_LIB_VERSION),
        ("libllama.so", LLAMA_LIB_VERSION),
        ("libggml.so", GGML_LIB_VERSION),
        ("libggml-base.so", GGML_LIB_VERSION),
        ("libggml-cpu.so", GGML_LIB_VERSION),
        ("libggml-cuda.so", GGML_LIB_VERSION),
        ("libmtmd.so", LLAMA_LIB_VERSION),
    ]
    for name, version in LIBS:
        versioned = f"{name}.{version}"
        for target in (name, f"{name}.0"):
            try:
                os.remove(target)
            except FileNotFoundError:
                pass
            os.symlink(versioned, target)
    print("CUDA SYMLINKS FIXED")
else:
    print("Skipping symlink fix (FIX_SYMLINKS is False)")


In [ ]:
# CELL 3
import os
import stat
SERVER = f"{LLAMA_DIR}/llama-server"
st = os.stat(SERVER)
os.chmod(SERVER, st.st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
print("EXECUTABLE:", os.access(SERVER, os.X_OK))


In [ ]:
# CELL 4
import os
import subprocess
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join([LLAMA_DIR] + CUDA_LIB_PATHS)
result = subprocess.run(
    [f"{LLAMA_DIR}/llama-server", "--version"],
    env=env, capture_output=True, text=True
)
print(result.returncode)
print(result.stdout)


In [ ]:
# CELL 5A
import os
import subprocess
import time

subprocess.run("pkill -9 llama-server", shell=True)
time.sleep(3)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join(LIB_PATHS)

print(f"Booting Trinity-Talk (7B) on Port {PORT_TALK}...")
proc_talk = subprocess.Popen([
    SERVER_BIN, "-m", os.path.join(MODEL_DIR, MODEL_TALK), "--alias", "trinity-talk",
    "--host", "127.0.0.1", "--port", str(PORT_TALK),
    "--ctx-size", "8192", "--jinja", "-ngl", "999", "--no-mmap", "--no-webui"
], env=env)

print(f"Booting Trinity-Chat (9B Vision) on Port {PORT_VISION}...")
proc_vision = subprocess.Popen([
    SERVER_BIN, "-m", os.path.join(MODEL_DIR, MODEL_VISION), "--mmproj", os.path.join(MODEL_DIR, MMPROJ_VISION),
    "--alias", "trinity-chat-vision", "--host", "127.0.0.1", "--port", str(PORT_VISION),
    "--ctx-size", "8192", "--jinja", "-ngl", "999", "--no-mmap", "--no-webui"
], env=env)

print("FAST MODE INITIATED! Run Cell 6 to wait for readiness.")


In [ ]:
# CELL 5B
import os
import subprocess
import time

subprocess.run("pkill -9 llama-server", shell=True)
time.sleep(3)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join(LIB_PATHS)

print(f"Booting Trinity-Deep (40B Vision) on Port {PORT_DEEP}...")
proc_deep = subprocess.Popen([
    SERVER_BIN, "-m", os.path.join(MODEL_DIR, MODEL_DEEP), "--mmproj", os.path.join(MODEL_DIR, MMPROJ_DEEP),
    "--alias", "trinity-deep-vision", "--host", "127.0.0.1", "--port", str(PORT_DEEP),
    "--ctx-size", "8192", "--jinja", "-ngl", "999", "--no-mmap", "--no-webui"
], env=env)

print("DEEP MODE INITIATED! Run Cell 6 to wait for readiness.")


In [ ]:
# CELL 6
import requests
import time

ports = [PORT_TALK, PORT_VISION, PORT_DEEP]
while True:
    ready = False
    for port in ports:
        try:
            r = requests.get(f"http://127.0.0.1:{port}/health", timeout=2)
            if r.status_code == 200:
                print(f"MODEL ON PORT {port} READY")
                ready = True
        except requests.RequestException:
            pass

    if ready:
        print("ACTIVE MODELS ARE READY!")
        break
    print("waiting for models to load...")
    time.sleep(10)


In [ ]:
# CELL 7
import requests
for port in [PORT_TALK, PORT_VISION, PORT_DEEP]:
    try:
        print(f"\n--- PORT {port} ---")
        print("HEALTH:", requests.get(f"http://127.0.0.1:{port}/health", timeout=2).text)
    except requests.RequestException:
        print(f"Port {port} is offline (Expected if not in this mode).")


In [ ]:
# CELL 8
import subprocess
subprocess.run(
    "pip install -q open-webui==0.9.6 fastapi uvicorn 'diffusers>=0.25.0' transformers accelerate torch omegaconf peft",
    shell=True
)
print("ALL DEPENDENCIES INSTALLED (PINNED TO FLAWLESS V0.9.6).")


In [ ]:
# CELL 9
import os
import subprocess
import time

subprocess.run("pkill -9 -f open-webui", shell=True)
subprocess.run("pkill -9 -f gunicorn", shell=True)
time.sleep(2)

env = os.environ.copy()
os.makedirs(f"{WORKDIR}/hf_cache", exist_ok=True)
env["HF_HOME"] = f"{WORKDIR}/hf_cache"
env["OPENAI_API_BASE_URLS"] = f"http://127.0.0.1:{PORT_TALK}/v1;http://127.0.0.1:{PORT_VISION}/v1;http://127.0.0.1:{PORT_DEEP}/v1"
env["OPENAI_API_KEYS"] = "dummy1;dummy2;dummy3"
env["WEBUI_AUTH"] = "True" if WEBUI_REQUIRE_LOGIN else "False"
env["RAG_EMBEDDING_ENGINE"] = ""
env["RAG_EMBEDDING_MODEL"] = "BAAI/bge-m3"
env["ENABLE_RAG_WEB_SEARCH"] = "True"
env["RAG_WEB_SEARCH_ENGINE"] = "duckduckgo"
env["RAG_WEB_SEARCH_RESULT_COUNT"] = "3"
env["ENABLE_RAG_LOCAL_WEB_FETCH"] = "True"
env["RAG_RERANKING_MODEL"] = "BAAI/bge-reranker-large"
env["DATA_DIR"] = f"{WORKDIR}/owui_data"

webui = subprocess.Popen(
    ["/usr/local/bin/open-webui", "serve", "--host", "0.0.0.0", "--port", str(PORT_WEBUI)],
    env=env
)
print("OPENWEBUI PID:", webui.pid)


In [ ]:
# CELL 10
import subprocess
import time
import re

subprocess.run("pkill -9 cloudflared", shell=True)
time.sleep(1)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared", shell=True)
subprocess.run("chmod +x cloudflared", shell=True)
print("Starting Cloudflare Tunnel...")

subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT_WEBUI}"],
    stderr=open('cloudflared.log', 'w')
)

print("\n=== YOUR OPEN WEBUI LINK ===")
link_found = False
for _ in range(20):
    time.sleep(1)
    try:
        with open('cloudflared.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                print(match.group(0))
                link_found = True
                break
    except FileNotFoundError:
        pass

if not link_found:
    print("Link not found yet. Run !cat cloudflared.log to check for errors.")
print("============================\n")


In [ ]:
# CELL 11
import subprocess
import time
import gc
import torch
import os

subprocess.run("pkill -9 -f image_api.py", shell=True)
gc.collect()
torch.cuda.empty_cache()

server_code = '''
import gc
import json
import torch
import traceback
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
import uvicorn
import base64
from io import BytesIO

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

MODEL_PATH = "__MODEL_PATH__"
pipe = None

print("Loading CyberRealistic V4.1...")
try:
    pipe = StableDiffusionPipeline.from_single_file(
        MODEL_PATH, torch_dtype=torch.float16, use_safetensors=True, safety_checker=None, requires_safety_checker=False
    )
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

    target_gpu = 0
    if torch.cuda.device_count() > 1:
        free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
        target_gpu = free.index(max(free))

    pipe = pipe.to(f"cuda:{target_gpu}")
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()
    gc.collect()
    torch.cuda.empty_cache()
    print("OK: CyberRealistic loaded")
except Exception:
    print("FAILED TO LOAD MODEL:")
    traceback.print_exc()
    pipe = None

@app.get("/health")
async def health(): return {"status": "ready" if pipe is not None else "model_not_loaded"}
@app.get("/sdapi/v1/sd-models")
async def get_models(): return [{"title": "CyberRealistic_V4.1", "model_name": "CyberRealistic_V4.1"}]
@app.get("/sdapi/v1/options")
async def get_options(): return {"sd_model_checkpoint": "CyberRealistic_V4.1"}
@app.post("/sdapi/v1/options")
async def set_options(request: Request): return {}
@app.get("/sdapi/v1/samplers")
async def get_samplers(): return [{"name": "Euler a", "aliases": ["euler_a"], "options": {}}]

@app.post("/sdapi/v1/txt2img")
async def txt2img(request: Request):
    if pipe is None: return {"images": [], "parameters": {}, "info": json.dumps({"error": "Model not loaded"})}
    try:
        gc.collect()
        torch.cuda.empty_cache()
        data = await request.json()
        prompt = data.get("prompt", "")
        negative_prompt = data.get("negative_prompt", "")
        steps = int(data.get("steps", 25))
        width = int(data.get("width", 512))
        height = int(data.get("height", 768))

        with torch.inference_mode():
            result = pipe(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=steps, width=width, height=height, guidance_scale=5.0)
            image = result.images[0]

        buf = BytesIO()
        image.save(buf, format="JPEG", quality=95)
        img_b64 = base64.b64encode(buf.getvalue()).decode()
        del result, image, buf
        gc.collect()
        torch.cuda.empty_cache()
        return {"images": [img_b64], "parameters": {}, "info": json.dumps({"prompt": prompt, "seed": -1, "steps": steps})}
    except Exception as e:
        return {"images": [], "parameters": {}, "info": json.dumps({"error": str(e)})}

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=__PORT__)
'''

server_code = server_code.replace("__MODEL_PATH__", os.path.join(MODEL_DIR, MODEL_CYBERREALISTIC))
server_code = server_code.replace("__PORT__", str(PORT_IMAGE))

with open("image_api.py", "w") as f:
    f.write(server_code)

print("Starting CyberRealistic server...")
proc = subprocess.Popen(["python", "image_api.py"], stdout=open("image_server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
print(f"Server booting (PID {proc.pid})")


In [ ]:
# CELL 12
import subprocess
import time
import gc
import torch
import os

subprocess.run("pkill -9 -f image_api.py", shell=True)
gc.collect()
torch.cuda.empty_cache()

server_code = '''
import gc
import json
import torch
import traceback
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
import uvicorn
import base64
from io import BytesIO

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

MODEL_PATH = "__MODEL_PATH__"
pipe = None

print("Loading RealVisXL V5.0 Lightning...")
try:
    # low_cpu_mem_usage=True protects system RAM during load
    pipe = StableDiffusionXLPipeline.from_single_file(
        MODEL_PATH,
        torch_dtype=torch.float16,
        use_safetensors=True,
        low_cpu_mem_usage=True
    )
    pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")

    target_gpu = 0
    if torch.cuda.device_count() > 1:
        free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
        target_gpu = free.index(max(free))

    free_vram = torch.cuda.mem_get_info(target_gpu)[0]
    if free_vram >= 9 * 1024 ** 3:
        pipe = pipe.to(f"cuda:{target_gpu}")
        try:
            pipe.enable_xformers_memory_efficient_attention()
        except Exception:
            pipe.enable_attention_slicing()
            pipe.enable_vae_slicing()
    else:
        pipe.enable_model_cpu_offload(gpu_id=target_gpu)

    gc.collect()
    torch.cuda.empty_cache()
    print("OK: RealVisXL loaded")
except Exception:
    print("FAILED TO LOAD MODEL:")
    traceback.print_exc()
    pipe = None

@app.get("/health")
async def health(): return {"status": "ready" if pipe is not None else "model_not_loaded"}
@app.get("/sdapi/v1/sd-models")
async def get_models(): return [{"title": "RealVisXL_V5.0_Lightning", "model_name": "RealVisXL_V5.0_Lightning"}]
@app.get("/sdapi/v1/options")
async def get_options(): return {"sd_model_checkpoint": "RealVisXL_V5.0_Lightning"}
@app.post("/sdapi/v1/options")
async def set_options(request: Request): return {}
@app.get("/sdapi/v1/samplers")
async def get_samplers(): return [{"name": "Euler", "aliases": ["euler"], "options": {}}]

@app.post("/sdapi/v1/txt2img")
async def txt2img(request: Request):
    if pipe is None: return {"images": [], "parameters": {}, "info": json.dumps({"error": "Model not loaded"})}
    try:
        gc.collect()
        torch.cuda.empty_cache()
        data = await request.json()
        prompt = data.get("prompt", "")
        negative_prompt = data.get("negative_prompt", "") or None
        steps = max(4, min(int(data.get("steps", 6)), 8))
        width = int(data.get("width", 1024))
        height = int(data.get("height", 1024))

        with torch.inference_mode():
            result = pipe(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=steps, width=width, height=height, guidance_scale=2.0)
            image = result.images[0]

        buf = BytesIO()
        image.save(buf, format="JPEG", quality=95)
        img_b64 = base64.b64encode(buf.getvalue()).decode()
        del result, image, buf
        gc.collect()
        torch.cuda.empty_cache()
        return {"images": [img_b64], "parameters": {}, "info": json.dumps({"prompt": prompt, "seed": -1, "steps": steps})}
    except Exception as e:
        return {"images": [], "parameters": {}, "info": json.dumps({"error": str(e)})}

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=__PORT__)
'''

server_code = server_code.replace("__MODEL_PATH__", os.path.join(MODEL_DIR, MODEL_REALVISXL))
server_code = server_code.replace("__PORT__", str(PORT_IMAGE))

with open("image_api.py", "w") as f:
    f.write(server_code)

print("Starting RealVisXL server...")
proc = subprocess.Popen(["python", "image_api.py"], stdout=open("image_server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(10)
print(f"Server booting (PID {proc.pid})")


In [ ]:
# DIAGNOSTIC CELL - READ IMAGE SERVER LOGS
import os
if os.path.exists("image_server.log"):
    with open("image_server.log", "r") as f:
        print("".join(f.readlines()[-40:]))
else:
    print("Log file not found.")
